In [7]:
url = "https://sofascore.p.rapidapi.com/tournaments/get-next-matches?tournamentId=17&seasonId=76986&pageIndex=0"
data = requests.get(url, headers=HEADERS, timeout=30).json()
events = data.get("events", [])
print(f"{len(events)} wedstrijden")
for e in events[:5]:
    status = e.get("status", {}).get("description")
    home   = e.get("homeTeam", {}).get("name")
    away   = e.get("awayTeam", {}).get("name")
    rnd    = e.get("roundInfo", {}).get("round")
    print(f"  GW{rnd} | {home} vs {away} | status: {status}")

30 wedstrijden
  GW29 | Manchester City vs Nottingham Forest | status: Not started
  GW29 | Brighton & Hove Albion vs Arsenal | status: Not started
  GW29 | Aston Villa vs Chelsea | status: Not started
  GW29 | Fulham vs West Ham United | status: Not started
  GW29 | Newcastle United vs Manchester United | status: Not started


In [8]:
"""
STAP 3 – Toekomstige wedstrijden + verwachte lineups
=====================================================
- Haalt aankomende Premier League wedstrijden op via get-next-matches
- Voegt ze toe aan 2025-2026_raw.csv (status='Not started', geen goals/stats)
- Maakt verwachte lineup op basis van laatste gespeelde opstelling per team
- Voegt die toe aan 2025-2026_players.csv (substitute='expected')
- Bij elke run: controleert of Sofascore al een echte lineup heeft
  (~1u voor aftrap) en overschrijft dan de placeholder
"""

import os
import csv
import time
import requests
import pandas as pd

# ── Configuratie ─────────────────────────────────────────────────────────────
API_KEY     = "2b277abcd2msh0e5627048810020p119057jsn4412b05235c9"
SEASON_ID   = 76986
TOURNAMENT  = 17
SEASON_STR  = "2025-2026"

MATCH_PATH  = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\matches\2025-2026_raw.csv"
PLAYER_PATH = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\players\2025-2026_players.csv"

DELAY   = 0.3
RETRIES = 3

HEADERS = {
    "x-rapidapi-key":  API_KEY,
    "x-rapidapi-host": "sofascore.p.rapidapi.com"
}

# ── Hulpfuncties ──────────────────────────────────────────────────────────────

def api_get(url: str) -> dict | None:
    for attempt in range(RETRIES):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            data = resp.json()
            if "message" in data:
                print(f"\n  ⛔ QUOTA OP: {data['message']}")
                return None
            return data
        except Exception as e:
            wait = 2 ** attempt
            print(f"  ⏳ Fout ({e}), wacht {wait}s...", end=" ", flush=True)
            time.sleep(wait)
    return None


def load_existing_ids(path: str) -> set:
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("match_id"):
                done.add(int(row["match_id"]))
    return done


def load_columns(path: str) -> list:
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        return next(csv.reader(f))


# ── Aankomende wedstrijden ophalen ────────────────────────────────────────────

def fetch_upcoming_events() -> list[dict]:
    print("   📋 Aankomende wedstrijden ophalen...")
    all_events = []

    for page in range(5):
        url = (f"https://sofascore.p.rapidapi.com/tournaments/get-next-matches"
               f"?tournamentId={TOURNAMENT}&seasonId={SEASON_ID}&pageIndex={page}")
        data = api_get(url)
        if not data:
            break
        events = data.get("events", [])
        if not events:
            break

        for e in events:
            all_events.append({
                "event_id":     e["id"],
                "round":        e.get("roundInfo", {}).get("round"),
                "status":       e.get("status", {}).get("description", ""),
                "home_team":    e.get("homeTeam", {}).get("name"),
                "away_team":    e.get("awayTeam", {}).get("name"),
                "home_team_id": e.get("homeTeam", {}).get("id"),
                "away_team_id": e.get("awayTeam", {}).get("id"),
                "timestamp":    e.get("startTimestamp"),
            })

        print(f"   Pagina {page}: {len(events)} wedstrijden")
        time.sleep(DELAY)

    print(f"   📊 Totaal aankomend: {len(all_events)}")
    return all_events


# ── Match schrijven ───────────────────────────────────────────────────────────

def write_upcoming_match(event: dict, match_cols: list, path: str):
    row = {col: "" for col in match_cols}
    row["match_id"]     = event["event_id"]
    row["season"]       = SEASON_STR
    row["round"]        = event["round"]
    row["timestamp"]    = event["timestamp"]
    row["status"]       = "Not started"
    row["home_team"]    = event["home_team"]
    row["away_team"]    = event["away_team"]
    row["home_team_id"] = event["home_team_id"]
    row["away_team_id"] = event["away_team_id"]

    file_exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=match_cols, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


# ── Lineup ophalen van Sofascore ──────────────────────────────────────────────

def fetch_sofascore_lineup(event_id: int) -> dict | None:
    """Probeer echte lineup op te halen — werkt alleen ~1u voor aftrap."""
    data = api_get(f"https://sofascore.p.rapidapi.com/matches/get-lineups?matchId={event_id}")
    time.sleep(DELAY)
    if not data:
        return None
    home_players = data.get("home", {}).get("players", [])
    away_players = data.get("away", {}).get("players", [])
    if home_players or away_players:
        return data
    return None


# ── Laatste bekende lineup per team ──────────────────────────────────────────

def get_last_lineup(players_df: pd.DataFrame, team: str) -> pd.DataFrame:
    """Haal de laatste basiself op voor een team."""
    # Wedstrijden waar dit team speelde
    mask = (players_df["home_team"] == team) | (players_df["away_team"] == team)
    team_rows = players_df[mask].copy()

    # Bepaal de juiste side
    team_rows["_team_side"] = team_rows.apply(
        lambda r: "home" if r["home_team"] == team else "away", axis=1
    )
    team_rows = team_rows[team_rows["side"] == team_rows["_team_side"]]

    # Alleen echte gespeelde wedstrijden (niet eerder toegevoegde expected)
    team_rows = team_rows[team_rows["substitute"] != "expected"]

    if team_rows.empty:
        return pd.DataFrame()

    # Laatste wedstrijd, alleen basiself
    last_match_id = team_rows.sort_values("timestamp").iloc[-1]["match_id"]
    lineup = team_rows[
        (team_rows["match_id"] == last_match_id) &
        (team_rows["substitute"].astype(str).str.lower() == "false")
    ].copy()

    return lineup


# ── Lineup schrijven ──────────────────────────────────────────────────────────

def write_lineup_rows(rows: list[dict], player_cols: list, path: str):
    if not rows:
        return
    file_exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=player_cols, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def build_expected_lineup(event: dict, lineup: pd.DataFrame,
                           side: str, player_cols: list) -> list[dict]:
    """Bouw placeholder lineup rijen op basis van laatste opstelling."""
    rows = []
    for _, player in lineup.iterrows():
        row = {col: "" for col in player_cols}
        # Kopieer spelerinfo
        for col in ["player_id", "player_name", "short_name", "position",
                    "shirt_number", "nationality", "height", "market_value",
                    "formation", "captain"]:
            if col in player.index:
                row[col] = player[col]
        row["match_id"]   = event["event_id"]
        row["season"]     = SEASON_STR
        row["round"]      = event["round"]
        row["timestamp"]  = event["timestamp"]
        row["home_team"]  = event["home_team"]
        row["away_team"]  = event["away_team"]
        row["side"]       = side
        row["substitute"] = "expected"   # marker: verwachte lineup
        rows.append(row)
    return rows


def build_real_lineup(event: dict, lineups_raw: dict,
                      player_cols: list) -> list[dict]:
    """Bouw echte lineup rijen op basis van Sofascore data."""
    rows = []
    for side in ["home", "away"]:
        formation = lineups_raw.get(side, {}).get("formation")
        for p in lineups_raw.get(side, {}).get("players", []):
            player = p.get("player", {})
            row = {col: "" for col in player_cols}
            row["match_id"]     = event["event_id"]
            row["season"]       = SEASON_STR
            row["round"]        = event["round"]
            row["timestamp"]    = event["timestamp"]
            row["home_team"]    = event["home_team"]
            row["away_team"]    = event["away_team"]
            row["side"]         = side
            row["formation"]    = formation
            row["player_id"]    = player.get("id")
            row["player_name"]  = player.get("name")
            row["short_name"]   = player.get("shortName")
            row["position"]     = p.get("position")
            row["shirt_number"] = p.get("shirtNumber")
            row["substitute"]   = p.get("substitute", False)
            row["captain"]      = p.get("captain", False)
            row["nationality"]  = player.get("country", {}).get("name")
            row["height"]       = player.get("height")
            row["market_value"] = (player.get("proposedMarketValueRaw", {}).get("value")
                                   if player.get("proposedMarketValueRaw") else "")
            rows.append(row)
    return rows


def remove_expected_lineup(event_id: int, path: str):
    """Verwijder placeholder lineup zodra echte beschikbaar is."""
    if not os.path.exists(path):
        return
    df = pd.read_csv(path, low_memory=False)
    before = len(df)
    df = df[~((df["match_id"] == event_id) & (df["substitute"].astype(str) == "expected"))]
    df.to_csv(path, index=False)
    print(f"   🗑️  {before - len(df)} placeholder rijen verwijderd", end=" ")


# ── Main ──────────────────────────────────────────────────────────────────────

def update_upcoming():
    print("\n══════════════════════════════════════════")
    print("  STAP 3 – Aankomende wedstrijden + lineups")
    print(f"  Seizoen: {SEASON_STR}")
    print("══════════════════════════════════════════")

    existing_ids = load_existing_ids(MATCH_PATH)
    match_cols   = load_columns(MATCH_PATH)
    player_cols  = load_columns(PLAYER_PATH)

    print(f"   📂 Bestaande wedstrijden: {len(existing_ids)}")

    # Laad player data voor laatste lineups
    players_df = pd.read_csv(PLAYER_PATH, low_memory=False)

    upcoming = fetch_upcoming_events()
    new_matches  = 0
    real_lineups = 0
    exp_lineups  = 0

    for event in upcoming:
        event_id  = event["event_id"]
        home_team = event["home_team"]
        away_team = event["away_team"]
        rnd       = event["round"]

        if event_id in existing_ids:
            # Wedstrijd al bekend — check of echte lineup nu beschikbaar is
            print(f"   🔄 GW{rnd} {home_team} vs {away_team}...", end=" ", flush=True)
            real = fetch_sofascore_lineup(event_id)
            if real:
                remove_expected_lineup(event_id, PLAYER_PATH)
                rows = build_real_lineup(event, real, player_cols)
                write_lineup_rows(rows, player_cols, PLAYER_PATH)
                print("✅ Echte lineup opgeslagen!")
                real_lineups += 1
            else:
                print("⏭️  nog geen lineup")
            continue

        # Nieuwe wedstrijd
        print(f"   ➕ GW{rnd} {home_team} vs {away_team}...", end=" ", flush=True)
        write_upcoming_match(event, match_cols, MATCH_PATH)
        new_matches += 1

        # Probeer eerst echte lineup
        real = fetch_sofascore_lineup(event_id)
        if real:
            rows = build_real_lineup(event, real, player_cols)
            write_lineup_rows(rows, player_cols, PLAYER_PATH)
            print("✅ Echte lineup!")
            real_lineups += 1
        else:
            # Gebruik laatste bekende opstelling als placeholder
            home_lineup = get_last_lineup(players_df, home_team)
            away_lineup = get_last_lineup(players_df, away_team)
            rows  = build_expected_lineup(event, home_lineup, "home", player_cols)
            rows += build_expected_lineup(event, away_lineup, "away", player_cols)
            write_lineup_rows(rows, player_cols, PLAYER_PATH)
            n = len(rows)
            print(f"📋 Verwachte lineup ({n} spelers)")
            exp_lineups += 1

        time.sleep(DELAY)

    print(f"\n   ✅ Klaar!")
    print(f"   📅 Nieuwe wedstrijden:  {new_matches}")
    print(f"   ✅ Echte lineups:       {real_lineups}")
    print(f"   📋 Verwachte lineups:   {exp_lineups}")
    print("══════════════════════════════════════════\n")


if __name__ == "__main__":
    update_upcoming()


══════════════════════════════════════════
  STAP 3 – Aankomende wedstrijden + lineups
  Seizoen: 2025-2026
══════════════════════════════════════════
   📂 Bestaande wedstrijden: 285
   📋 Aankomende wedstrijden ophalen...
   Pagina 0: 30 wedstrijden
   Pagina 1: 30 wedstrijden
   Pagina 2: 30 wedstrijden
   Pagina 3: 10 wedstrijden
   📊 Totaal aankomend: 100
   ➕ GW29 Manchester City vs Nottingham Forest... ✅ Echte lineup!
   ➕ GW29 Brighton & Hove Albion vs Arsenal... ✅ Echte lineup!
   ➕ GW29 Aston Villa vs Chelsea... ✅ Echte lineup!
   ➕ GW29 Fulham vs West Ham United... ✅ Echte lineup!
   ➕ GW29 Newcastle United vs Manchester United... ✅ Echte lineup!
   ➕ GW29 Tottenham Hotspur vs Crystal Palace... ✅ Echte lineup!
   ➕ GW30 Sunderland vs Brighton & Hove Albion... 📋 Verwachte lineup (22 spelers)
   ➕ GW30 Brentford vs Wolverhampton... 📋 Verwachte lineup (22 spelers)
   ➕ GW30 Burnley vs Bournemouth... 📋 Verwachte lineup (22 spelers)
   ➕ GW30 Arsenal vs Everton... 📋 Verwachte line